# Fine-tuning the model

In [17]:
# pip install --upgrade jupyter jupyterlab notebook ipywidgets
# pip install accelerate

In [25]:
from tqdm import tqdm
import numpy as np

In [19]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding


raw_datasets = load_dataset("SetFit/mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Pair Tokenizer Function
def tokenize_function(example):
    return tokenizer(example["text1"], example["text2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, 
                                      batched=True,
                                      num_proc=10)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Repo card metadata block was not found. Setting CardData to empty.


### 1. Define a **TrainingArguments** class that will contain all the hyperparameters the **Trainer** will use for training and evaluation

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer",
                                  per_device_train_batch_size=32,
                                  fp16=True, # Enable mixed precision - faster training and reduced memory usage
                                  dataloader_num_workers=8)

### 2. Define the Model.
#### 2.1 - Use **AutoModelForSequenceClassification** class with *two* labels

In [21]:
from transformers import Trainer
from transformers import AutoModelForSequenceClassification

**Note**: BERT has not been pretrained on classifying pairs of sentences, so the head of the pretrained model has been discarded and a new head suitable for sequence classification has been added instead. The warnings indicate that some weights were not used (the ones corresponding to the dropped pretraining head) and that some others were randomly initialized (the ones for the new head).

In [22]:
#use the AutoModelForSequenceClassification class, with two labels
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 2.2 Define a Trainer by passing it all the objects constructed up to now — the **model**, the **training_args**, the **training and validation datasets**, our **data_collator**, and our **processing_class**. 

The **processing_class** parameter tells the Trainer which tokenizer to use for processing.

The **eval_dataset** parameter is *optional*, You won't get evaluation metrics during training, but training will still work.

In [23]:
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator, # Default collator: DataCollatorWithPadding
    processing_class=tokenizer,
)

#### 2.3 - To fine-tune the model on our dataset, call the **train()** method of the **Trainer**

##### This will start the fine-tuning (which should take a couple of minutes on a GPU) and report the training loss every 500 steps.

##### It won’t, however, tell you how well (or badly) your model is performing. This is because:

1. We didn’t tell the Trainer to evaluate during training by setting eval_strategy in TrainingArguments to either "steps" (evaluate every eval_steps) or "epoch" (evaluate at the end of each epoch).
2. We didn’t provide the Trainer with a compute_metrics() function to calculate a metric during said evaluation (otherwise the evaluation would just have printed the loss, which is not a very intuitive number).

In [24]:
trainer.train()
# Try #1: Estimated 39 minutes to complete

c:\Users\einob\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=345, training_loss=0.3380662779877151, metrics={'train_runtime': 1471.6119, 'train_samples_per_second': 7.478, 'train_steps_per_second': 0.234, 'total_flos': 450668126729280.0, 'train_loss': 0.3380662779877151, 'epoch': 3.0})

## 3. Evaluate the trained model

3.1. We can evaluate the model by creating a **compute_metrics()** function and use it the next time we train. 

The function must take an **EvalPrediction** object (which is a named tuple with a **predictions & label_ids fields**) and will return a dictionary mapping strings to floats (the strings being the names of the metrics returned, and the floats their values). 

**Trainer.predict()** command: use to get some predictions from the model. the output is a tuple with 3 fields:
1. predictions - 2D array. ex. (# of elements in dataset, 2)
2. label_ids
3. metrics - contains the **loss on the dataset passed** & some **time metrics** (how long it took to predict, in total and on average)

In [26]:
# get some predictions from our model
predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)

c:\Users\einob\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


(408, 2) (408,)


#### 3.2. We need predictions that we can compare to our labels. We must transform the **predictions** by taking the index with the maximum value on the second axis.

In [27]:
preds = np.argmax(predictions.predictions, axis=-1)

#### 3.3. After this we can do comparison with *preds* to the labels

the *cased* BERT model used below has an **accuracy of 87.01%** on the validation set and an **F1 score of 90.94%**. 

Those are the two metrics used to evaluate results on the MRPC dataset for the GLUE benchmark. For the *uncased* base model in the BERT paper, there was a **F1 score of 88.9**

In [ ]:
# pip install evaluate
import evaluate

metric = evaluate.load("glue", "mrpc") # instantiate an evaluation module
metric.compute(
    predictions=preds,
    references=predictions.label_ids
)


{'accuracy': 0.8700980392156863, 'f1': 0.9094017094017094}

Function for computing metrics. This will be used within a more dynamic Trainer

In [35]:
def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

Let's retrain the dataset with a new training_args with the same speedy parameters as before but now with an **epoch evaluation strategy** and the use of *compute_metrics()* function to get more enhanced output metrics parameters. 

**eval_strategy="epoch"**  reports the validation loss and metrics at the end of each epoch on top of the training loss. eval_strategy can be set to **'epoch'**, **'steps'**, or **'no'** to control evaluation timing.

In [ ]:
#Put it all together
# speedy training arguments with the epoch evaluation strategry included
training_args = TrainingArguments(
    "test-trainer", 
    per_device_train_batch_size=32,
    fp16=True,  # Enable mixed precision - faster training and reduced memory usage
    dataloader_num_workers=8,
    eval_strategy="epoch",

    ## There are many types of trainers that can be included:
    #per_device_train_batch_size=4,
    #gradient_accumulation_steps=4,  
        ## For effective larger batch sizes when GPU memory is limited: 
            ## Effective batch size = 4 * 4 = 16
        ## allows you to simulate larger batch sizes by 
        ## accumulating gradients over multiple forward passes.
    #learning_rate=2e-5,
    #lr_scheduler_type="cosine",  
        ## Learning Rate Scheduling # different schedulers to try here
    )

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

trainer = Trainer(
    model,
    training_args, #Same speedy arguments as before
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [37]:
trainer.train()

c:\Users\einob\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.372002,0.835784,0.879279
2,No log,0.336913,0.852941,0.894366
3,No log,0.425616,0.867647,0.907850


c:\Users\einob\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\einob\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\einob\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=345, training_loss=0.3389604651409647, metrics={'train_runtime': 1599.0156, 'train_samples_per_second': 6.882, 'train_steps_per_second': 0.216, 'total_flos': 450668126729280.0, 'train_loss': 0.3389604651409647, 'epoch': 3.0})

### Validation Results

| Epoch | Validation Loss | Accuracy | F1 Score |
|--------|----------------|----------|----------|
| 1 | 0.3720 | 83.58% | 87.93% |
| 2 | 0.3369 | 85.29% | 89.44% |
| 3 | 0.4256 | 86.76% | 90.79% |

**Observations**
- Validation accuracy increased from **83.58%** to **86.76%** over three epochs.
- The F1 score improved from **0.8793** to **0.9079**, indicating improved performance on the paraphrase classification task.
- Validation loss decreased between epochs 1 and 2 but increased in epoch 3, suggesting the onset of mild overfitting despite improvements in accuracy and F1 score.

In [1]:
import torch

print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.6.0+cu124
CUDA version: 12.4
CUDA available: True
